### Import Required Libraries

In [19]:
from minsearch import Index
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

### Fetch Data

In [20]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

### Questions 1 : How many lesson pages

In [21]:
print("Number of lesson pages: ", len(files))

Number of lesson pages:  72


### Questions 2 : Indexing and searching

In [22]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [23]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index = build_index(documents)

In [24]:
q2 = "How does the agentic loop keep calling the model until it stops?"

result = index.search(q2)[0]['filename']
print(result)

01-agentic-rag/lessons/14-agentic-loop.md


### Questions 3 : RAG

In [25]:
from codebase.rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [26]:
answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model in a `while True` loop and checks whether the response contains any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, it `break`s out of the loop.

So the stop condition is: **no function calls in the model’s response**.


In [27]:
print("Number of token used to send the request: ", usage.input_tokens)

Number of token used to send the request:  7126


### Questions 4 : Chunking

In [28]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Number of chunks: ", len(chunks))

Number of chunks:  295


### Questions 5 : RAG with chunking

In [29]:
index_chunk = build_index(chunks)

assistant = RAGBase(
    index=index_chunk,
    llm_client=openai_client,
)

answer_chunk, usage_chunk = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print("Number of token used to send the request: ", usage.input_tokens)
print("Number of token used to send the request with chunk: ", usage_chunk.input_tokens)
print("Diff (fewer): ", usage.input_tokens/usage_chunk.input_tokens )

Number of token used to send the request:  7126
Number of token used to send the request with chunk:  2309
Diff (fewer):  3.086184495452577


### Questions 6 : Turning it into an agent

In [30]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [31]:
def search(query: str) -> str:
    """
    Search the course lessons for information relevant to the query.
    """
    results = index_chunk.search(query, num_results=5)
    return assistant.build_context(results)

In [32]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons for information relevant to the query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [39]:
instructions="""You're a course teaching assistant. Answer the student's question using the
# search tool. Make multiple searches with different keywords before answering."""

In [40]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [41]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
